In [1]:
!pip install pyarrow fastparquet -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 33.3 MB/s eta 0:00:00


**Section 3.1 — Data Structures**
Pandas flips this completely. You work with entire columns at once.

Not one customer — all 10,000 customers in one operation. That shift is what makes data processing fast.

In [ ]:
import pandas as pd

amount = pd.Series([15000, 4000, 8000, 22000, 3000])
print(amount)

print(amount * 1.18)
print(amount > 10000)
print(amount.sum())
print(amount.mean())

0    15000
1     4000
2     8000
3    22000
4     3000
dtype: int64
0    17700.0
1     4720.0
2     9440.0
3    25960.0
4     3540.0
dtype: float64
0     True
1    False
2    False
3     True
4    False
dtype: bool
52000
10400.0


In [ ]:
revenue = pd.Series(
    [45000, 12000, 72000, 31000],
    index = ["Priya", "Sneha", "Rahul", "Divya"]
)

print(revenue)
print(revenue["Rahul"])
print(revenue[revenue > 30000])

Priya    45000
Sneha    12000
Rahul    72000
Divya    31000
dtype: int64
72000
Priya    45000
Rahul    72000
Divya    31000
dtype: int64


**DataFrame — a full table**

A DataFrame is a collection of Series — one per column — all sharing the same index.

In [ ]:
customers = pd.DataFrame({
    "customer_id" : [301, 302, 303, 304, 305],
    "name"        : ["Priya", "Arjun", "Sneha", "Rahul", "Divya"],
    "spent"       : [45000, 4000, 12000, 72000, 31000],
    "status"      : ["active", "inactive", "active", "active", "inactive"]
})

print(customers)

   customer_id   name  spent    status
0          301  Priya  45000    active
1          302  Arjun   4000  inactive
2          303  Sneha  12000    active
3          304  Rahul  72000    active
4          305  Divya  31000  inactive


**The five commands you run on every new dataset**

Every single time you load data — before touching anything — run these five:

In [ ]:
print(customers.shape)
print(customers.dtypes)
print(customers.info())
print(customers.head(3))
print(customers.describe())

(5, 4)
customer_id     int64
name           object
spent           int64
status         object
dtype: object
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   customer_id  5 non-null      int64 
 1   name         5 non-null      object
 2   spent        5 non-null      int64 
 3   status       5 non-null      object
dtypes: int64(2), object(2)
memory usage: 292.0+ bytes
None
   customer_id   name  spent    status
0          301  Priya  45000    active
1          302  Arjun   4000  inactive
2          303  Sneha  12000    active
       customer_id         spent
count     5.000000      5.000000
mean    303.000000  32800.000000
std       1.581139  27160.633277
min     301.000000   4000.000000
25%     302.000000  12000.000000
50%     303.000000  31000.000000
75%     304.000000  45000.000000
max     305.000000  72000.000000


In [ ]:
orders = pd.DataFrame({
    "order_id"    : [101, 102, 103, 104, 105],
    "customer_id" : [301, 302, 301, 303, 302],
    "amount"      : [15000, 4000, 8000, 22000, 3000],
    "status"      : ["pending", "shipped", "delivered", "pending", "cancelled"],
    "order_date"  : ["2024-01-15", "2024-01-16", "2024-01-17", "2024-01-18", "2024-01-19"]
})

print(orders)

print(orders.shape)
print(orders.dtypes)
print(orders.info())
print(orders.head(3))
print(orders.describe())

   order_id  customer_id  amount     status  order_date
0       101          301   15000    pending  2024-01-15
1       102          302    4000    shipped  2024-01-16
2       103          301    8000  delivered  2024-01-17
3       104          303   22000    pending  2024-01-18
4       105          302    3000  cancelled  2024-01-19
(5, 5)
order_id        int64
customer_id     int64
amount          int64
status         object
order_date     object
dtype: object
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   order_id     5 non-null      int64 
 1   customer_id  5 non-null      int64 
 2   amount       5 non-null      int64 
 3   status       5 non-null      object
 4   order_date   5 non-null      object
dtypes: int64(3), object(2)
memory usage: 332.0+ bytes
None
   order_id  customer_id  amount     status  order_date
0       101          301   150

**Section 3.2 — Loading Data**
This is where real DE work starts.

You rarely create DataFrames manually like we just did.

You load them from files and databases.

In [ ]:
import pandas as pd
import json
import os

os.makedirs("tradesphere", exist_ok=True)

# Customers CSV
customers_data = """customer_id,name,email,country,spent,status,join_date
301,  Priya Mehta  ,priya@email.com,India,45000,active,2022-03-15
302,Arjun Patel,arjun@email.com,India,4000,inactive,2023-01-20
303,  Sneha Shah,sneha@email.com,UAE,12000,active,2022-07-08
304,Rahul Verma,rahul@email.com,India,72000,Active,2021-11-30
305,,divya@email.com,India,9000,active,2023-05-14
306,Divya Nair,divya2@email.com,UAE,31000,invalid,2022-09-01
307,Karan Singh,karan@email.com,India,bad_value,active,2023-02-28
308,Meera Iyer,meera@email.com,India,55000,active,2021-06-15"""

with open("tradesphere/customers.csv", "w") as f:
    f.write(customers_data)

# Orders JSON
orders_data = [
    {"order_id": 1001, "customer_id": 301, "amount": 15000,
     "status": "delivered", "order_date": "2024-01-15",
     "product_id": 501},
    {"order_id": 1002, "customer_id": 302, "amount": 4000,
     "status": "shipped", "order_date": "2024-01-16",
     "product_id": 502},
    {"order_id": 1003, "customer_id": 301, "amount": 8000,
     "status": "delivered", "order_date": "2024-02-10",
     "product_id": 503},
    {"order_id": 1004, "customer_id": 303, "amount": 22000,
     "status": "pending", "order_date": "2024-02-14",
     "product_id": 501},
    {"order_id": 1005, "customer_id": 304, "amount": 3000,
     "status": "cancelled", "order_date": "2024-03-01",
     "product_id": 504},
    {"order_id": 1006, "customer_id": 304, "amount": 45000,
     "status": "delivered", "order_date": "2024-03-15",
     "product_id": 502},
    {"order_id": 1007, "customer_id": 308, "amount": 12000,
     "status": "shipped", "order_date": "2024-03-20",
     "product_id": 503},
    {"order_id": 1008, "customer_id": 301, "amount": 9500,
     "status": "delivered", "order_date": "2024-04-02",
     "product_id": 501}
]

with open("tradesphere/orders.json", "w") as f:
    json.dump(orders_data, f, indent=2)

print("TradeSphere files created")
print(os.listdir("tradesphere"))

TradeSphere files created
['orders.json', 'customers.csv']


**Reading a CSV — with the parameters that matter**

Most tutorials show you pd.read_csv("file.csv") and nothing else.

In production that is never enough. Run this:

In [ ]:
customers = pd.read_csv(
    "tradesphere/customers.csv",
    dtype = {
        "customer_id" : str,
        "country"     : "category",
        "status"      : "category"
    },

)

print(customers)
print("\n")
print(customers.dtypes)
print("\n")
print(customers.isnull().sum())

  customer_id             name             email country      spent    status  \
0         301    Priya Mehta     priya@email.com   India      45000    active   
1         302      Arjun Patel   arjun@email.com   India       4000  inactive   
2         303       Sneha Shah   sneha@email.com     UAE      12000    active   
3         304      Rahul Verma   rahul@email.com   India      72000    Active   
4         305              NaN   divya@email.com   India       9000    active   
5         306       Divya Nair  divya2@email.com     UAE      31000   invalid   
6         307      Karan Singh   karan@email.com   India  bad_value    active   
7         308       Meera Iyer   meera@email.com   India      55000    active   

    join_date  
0  2022-03-15  
1  2023-01-20  
2  2022-07-08  
3  2021-11-30  
4  2023-05-14  
5  2022-09-01  
6  2023-02-28  
7  2021-06-15  


customer_id      object
name             object
email            object
country        category
spent            object
stat

**Reading JSON**

In [ ]:
orders = pd.read_json("tradesphere/orders.json")

print(orders)
print("\n")
print(orders.dtypes)

   order_id  customer_id  amount     status  order_date  product_id
0      1001          301   15000  delivered  2024-01-15         501
1      1002          302    4000    shipped  2024-01-16         502
2      1003          301    8000  delivered  2024-02-10         503
3      1004          303   22000    pending  2024-02-14         501
4      1005          304    3000  cancelled  2024-03-01         504
5      1006          304   45000  delivered  2024-03-15         502
6      1007          308   12000    shipped  2024-03-20         503
7      1008          301    9500  delivered  2024-04-02         501


order_id        int64
customer_id     int64
amount          int64
status         object
order_date     object
product_id      int64
dtype: object


Answer these three questions from the output:
 1. What are the dtypes of each column?
 2. How many null values are in each column?
 3. What does describe() tell you about the amount column?
 4. Min, max, mean — what does this suggest about the data?

In [ ]:
orders = pd.read_json("tradesphere/orders.json")

print(orders.dtypes)
print(orders.describe())
print(orders.isnull().sum())

order_id        int64
customer_id     int64
amount          int64
status         object
order_date     object
product_id      int64
dtype: object
         order_id  customer_id        amount  product_id
count     8.00000     8.000000      8.000000    8.000000
mean   1004.50000   303.000000  14812.500000  502.125000
std       2.44949     2.390457  13638.018027    1.125992
min    1001.00000   301.000000   3000.000000  501.000000
25%    1002.75000   301.000000   7000.000000  501.000000
50%    1004.50000   302.500000  10750.000000  502.000000
75%    1006.25000   304.000000  16750.000000  503.000000
max    1008.00000   308.000000  45000.000000  504.000000
order_id       0
customer_id    0
amount         0
status         0
order_date     0
product_id     0
dtype: int64


**Creating a Dataset: From the code**

In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import random
import os
import csv

# ── Setup ─────────────────────────────────────
np.random.seed(42)
random.seed(42)

base_path = "/content/tradesphere"
os.makedirs(base_path, exist_ok=True)

# ── Reference data ────────────────────────────
countries     = ["India", "UAE", "USA", "UK", "Singapore"]
categories    = ["Electronics", "Clothing", "Food", "Books", "Home"]
statuses      = ["active", "inactive"]
order_statuses= ["pending", "shipped", "delivered", "cancelled", "returned"]
pay_methods   = ["credit_card", "debit_card", "UPI", "net_banking", "wallet"]
warehouses    = ["Mumbai", "Delhi", "Dubai", "London", "Singapore"]

def random_date(start, end):
    return start + timedelta(days=random.randint(0, (end - start).days))

start_date = datetime(2022, 1, 1)
end_date   = datetime(2024, 12, 31)

# ── 1. Customers ──────────────────────────────
customers = []

for i in range(1000):
    cid        = 1000 + i
    fname      = random.choice(["Priya","Arjun","Sneha","Rahul","Divya"])
    lname      = random.choice(["Mehta","Patel","Shah","Verma","Nair"])
    country    = random.choice(countries)
    status     = random.choice(statuses)
    join_date  = random_date(start_date, end_date)
    total_spent= round(random.uniform(500, 200000), 2)

    name = f"{fname} {lname}"

    # Inject issues
    if i % 50 == 0:
        name = "  " + name + "  "
    if i % 75 == 0:
        name = ""
    if i % 100 == 0:
        total_spent = None
    if i % 120 == 0:
        status = random.choice(["Active","ACTIVE","Invalid"])

    customers.append({
        "customer_id": str(cid),
        "name": name,
        "total_spent": total_spent,
        "status": status,
        "join_date": join_date.strftime("%Y-%m-%d")
    })

pd.DataFrame(customers).to_csv(
    f"{base_path}/customers.csv",
    index=False,
    quoting=csv.QUOTE_ALL   # 🔥 FIX
)

# ── 2. Products ───────────────────────────────
products = []

for i in range(200):
    products.append({
        "product_id": str(5000 + i),
        "name": f"Product_{i}",
        "price": round(random.uniform(100, 50000), 2)
    })

pd.DataFrame(products).to_csv(
    f"{base_path}/products.csv",
    index=False,
    quoting=csv.QUOTE_ALL
)

# ── 3. Orders ─────────────────────────────────
orders = []

for i in range(5000):
    status = random.choice(order_statuses)
    order_date = random_date(start_date, end_date)
    amount = round(random.uniform(200, 80000), 2)

    if status in ["shipped","delivered"]:
        delivery_date = (order_date + timedelta(days=random.randint(1, 14))).strftime("%Y-%m-%d")
    else:
        delivery_date = None

    if i % 80 == 0:
        amount = None
    if i % 150 == 0:
        status = "bad_status"

    orders.append({
        "order_id": str(10000 + i),
        "amount": amount,
        "status": status,
        "order_date": order_date.strftime("%Y-%m-%d"),
        "delivery_date": delivery_date
    })

pd.DataFrame(orders).to_csv(
    f"{base_path}/orders.csv",
    index=False,
    quoting=csv.QUOTE_ALL   # 🔥 CRITICAL FIX
)

# ── 4. Payments ───────────────────────────────
payments = []

for order in orders:
    if random.random() > 0.05:
        pay_date = datetime.strptime(order["order_date"], "%Y-%m-%d") + timedelta(days=1)

        payments.append({
            "payment_id": f"PAY_{order['order_id']}",
            "order_id": order["order_id"],
            "amount": order["amount"],
            "status": "success",
            "payment_date": pay_date.strftime("%Y-%m-%d")
        })

pd.DataFrame(payments).to_csv(
    f"{base_path}/payments.csv",
    index=False,
    quoting=csv.QUOTE_ALL
)

# ── SAFE READ (FIXED) ─────────────────────────
print("\nTradeSphere dataset ready\n")

for f in os.listdir(base_path):
    path = f"{base_path}/{f}"

    df = pd.read_csv(
        path,
        engine="python",        # 🔥 FIX
        on_bad_lines="skip"     # 🔥 FIX
    )

    print(f"{f:20s} {len(df):>6} rows | {len(df.columns)} cols")


TradeSphere dataset ready

orders.json              64 rows | 1 cols
products.csv            200 rows | 3 cols
customers.csv          1000 rows | 5 cols
orders.csv             5000 rows | 5 cols
payments.csv           4751 rows | 5 cols


**Section 3.3 — Selecting and Filtering**

Load the data first
Run this at the top of your notebook. Every section from here uses these DataFrames:

In [ ]:
customers = pd.read_csv(
    "tradesphere/customers.csv",
    dtype      = {"customer_id": str},
    parse_dates= ["join_date"]
)
orders = pd.read_csv(
    "tradesphere/orders.csv",
    dtype      = {"customer_id": str, "order_id": str, "product_id": str},
    parse_dates= ["order_date"]
)

products = pd.read_csv(
    "tradesphere/products.csv",
    dtype = {"product_id": str}
)

print("customers:", customers.shape)
print("orders   :", orders.shape)
print("products :", products.shape)

customers: (1000, 5)
orders   : (5000, 5)
products : (200, 3)


**Selecting columns**

The most basic operation. Three ways to do it:

In [ ]:
# Single column — returns a Series
#print(customers["name"])

# Multiple columns — returns a DataFrame
print(customers[["name", "country", "status"]])

# All rows, specific columns — clean way
#subset = customers[["customer_id", "name", "total_spent"]]
#print(subset.head())


KeyError: "['country'] not in index"

**loc — selecting by label**

loc selects by row label and column name:

In [ ]:
# All rows, one column
#print(customers.loc[:, 'name'])

# All rows, multiple columns
print(customers.loc[:, ["name", "country", "total_spent"]])

# Specific rows by index label, all columns
#print(customers.loc[0:4, :])

# Specific rows AND specific columns
#print(customers.loc[0:4, ["name", "total_spent"]])

KeyError: "['country'] not in index"

**iloc — selecting by position**

iloc uses integer positions — row number and column number.

Zero indexed:

In [ ]:
# First row
#print(customers.iloc[0])

# First 5 rows
#print(customers.iloc[0:5])

# First 5 rows, first 3 columns
#print(customers.iloc[0:5, 0:3])

# Last 5 rows
print(customers.iloc[-5:])

**Boolean filtering — the most important pattern**

This is what you will use in every single pipeline.

The idea is simple — create a True/False mask, apply it to the DataFrame, keep only the True rows.

In [ ]:
# Step 1 — create the mask
mask = customers["status"] == "active"
#print(mask.head(10))

# Step 2 — apply the mask
active_customers = customers[mask]
print(active_customers.shape)
print(active_customers["status"].unique())

**Multiple conditions**

In [ ]:
from ast import And
# AND — both conditions must be true
# Parentheses around each condition are mandatory

india_active = customers[
    (customers["country"] == "India") &
    (customers["status"] == "active")
]
print(f"India active customers: {len(india_active)}")

# OR — either condition true

india_or_uae = customers[
    (customers["country"] == "India") |
    (customers["country"] == "UAE")
]
print(f"India or UAE: {len(india_or_uae)}")

# NOT — exclude a condition
not_cancelled = orders[
    orders["status"] != "cancelled"
]
print(f"Non-cancelled orders: {len(not_cancelled)}")

**isin() — filtering against a list of values**

In [ ]:
# Instead of multiple OR conditions
target_countries = ["India", "UAE", "Singapore"]

target = customers[
    customers["country"].isin(target_countries)
]
print(f"Target country customers: {len(target)}")

# Inverse — NOT in
other_countries = customers[
    ~customers["country"].isin(target_countries)
]
print(f"Other country customers: {len(other_countries)}")

**query() — readable filtering**

For complex conditions query() is cleaner to read:

In [ ]:
# Same as boolean filter but reads like English
high_value_india = customers.query(
    "country == 'India' and total_spent > 50000 and status == 'active'"
)
print(f"High value India customers: {len(high_value_india)}")

**Filtering on nulls**

In [ ]:
# Rows where total_spent is null
missing_spent = customers[customers["total_spent"].isna()]
print(f"Missing spent: {len(missing_spent)}")

# Rows where total_spent is NOT null
has_spent = customers[customers["total_spent"].notna()]
print(f"Has spent: {len(has_spent)}")

# Rows where name is empty string — different from null
empty_name = customers[customers["name"] == " "]
print(f"Empty name: {len(empty_name)}")

Using the orders DataFrame —

write filtering code that answers these three

**business questions:**
**Question 1**:

How many orders were delivered in 2024?

Hint — order_date is already parsed as datetime

Use order_date.dt.year to extract the year

In [ ]:
# Convert to datetime
order = pd.read_csv("tradesphere/orders.csv")
order['order_date'] = pd.to_datetime(order['order_date'])

# Filter
delivered_2024 = order[
    (order['order_date'].dt.year == 2024) &
    (order['status'] == 'delivered')
]

print(f"Number of Orders Delivered in 2024 is {len(delivered_2024)}")

 **Question 2**

 What is the total amount of high value pending orders?

 High value means amount > 30000

 Hint — filter first, then .sum()

In [ ]:
high_value_pending = order.query(
    "amount > 30000 and status == 'pending'"
)['amount'].sum()

print(high_value_pending)

 Question 3

 How many orders were placed by customers

 from this list: ["1005", "1010", "1025", "1050"]

 Use isin()

In [ ]:
customer_list = ["1005", "1010", "1025", "1050"]

filtered_orders = order[order['customer_id'].isin(customer_list)]

count = len(filtered_orders)

print(count)

**Section 3.4 — Indexing and MultiIndex**

What is an index
Every DataFrame has an index — it is the row label. By default Pandas assigns 0, 1, 2, 3... automatically. You have seen this in every output so far — the numbers on the left side.


But the index can be anything meaningful. A customer_id. A date. A product code. Setting a meaningful index makes lookups faster and makes your data easier to work with.

**set_index() — making a column the index**

In [ ]:
print(customers.head(3))
print(customers.index)

**Now set customer_id as the index:**

In [ ]:
customers_index = customers.set_index("customer_id")

print(customers_index.head(3))
print(customers_index.index)

**Now lookups become instant and readable:**

In [ ]:
# Get one specific customer by ID
#print(customers_index.loc["1005"])

# Get multiple specific customers
print(customers_index.loc[["1005","1006","1008"]])

**reset_index() — putting the index back as a column**

After operations like groupby, the result often has the groupby column as the index. You almost always want it back as a regular column:

In [ ]:
# This produces a DataFrame with country as the index
status_revenue = orders.groupby("status")["amount"].sum()
print(status_revenue)
print(type(status_revenue))

In [ ]:
#Now Reset:

status_revenue = status_revenue.reset_index()
print(status_revenue)
print(type(status_revenue))

**sort_index() and sort_values()**

In [ ]:
# Sort by index
print(customers_index.sort_index().head())

In [ ]:
# Sort by a column value
print(customers.sort_values("total_spent", ascending= False).head())

In [ ]:
# Sort by multiple columns

print(orders.sort_values(
    ["status", "amount"],
    ascending=[True, False]
).head(10))

**MultiIndex — hierarchical indexing**


When you do a groupby on two columns — say country and status — the result has two levels of labels. Country on the outside, status on the inside. That two-level structure is a MultiIndex.

In [ ]:
# Group by two columns

revenue_by_country_status = (
    orders.merge(
        customers[["customer_id", "country"]],
        on="customer_id"
    )
    .groupby(["country", "status"])["amount"]
    .sum()
)

print(revenue_by_country_status)
print(type(revenue_by_country_status))
print(revenue_by_country_status.index)

**Navigating a MultiIndex**

In [ ]:
print(revenue_by_country_status.loc["India"])

Access one specific country + status combination

In [ ]:
print(revenue_by_country_status.loc["India", "delivered"])

**Flattening a MultiIndex — the most used pattern**

In production you almost never want to keep a MultiIndex. You flatten it into a regular DataFrame with named columns:

In [ ]:
revenue_flat = (orders.merge(
    customers[["customer_id","country"]],
    on="customer_id"
  )
  .groupby(["country","status"])["amount"]
  .sum().reset_index()
)

revenue_flat.columns = ["country", "order_status", "total_revenue"]
print(revenue_flat)
print(revenue_flat.dtypes)

Using orders and customers DataFrames:
1. Set order_id as the index of orders DataFrame
  Look up order "10005" using loc

2. Find total amount per country per payment_method

    Steps:
    — merge orders with customers on customer_id
      (you need country from customers)

    — groupby country and payment_method

    — sum the amount

    — reset_index to get a flat DataFrame

    — sort by total amount descending
    
    — print top 10 rows

Set order_id as the index of orders DataFrame Look up order "10005" using loc

In [ ]:
order = pd.read_csv("/content/tradesphere/orders.csv")

order_index = order.set_index("order_id")
print(order_index.loc[[10005]])

print(order_index.index)




**Find total amount per country per payment_method**

In [ ]:
country_payment = (
    orders.merge(
        customers[["customer_id", "country"]],
        on="customer_id"
    )
    .groupby(["country", "payment_method"])["amount"]
    .sum()
    .reset_index()
)

country_payment.columns = ["country", "payment_method", "total_amount"]

country_payment = country_payment.sort_values(
    "total_amount",
    ascending=False
)

print(country_payment.head(10))

**Section 3.5 — Data Cleaning**

This is the most important section in Module 3. In every real pipeline, 60% of your work is cleaning. Raw data is always broken in the same ways — nulls, wrong types, inconsistent strings, duplicates. Let's fix all of them on real TradeSphere data.

First — understand what you have
Before cleaning anything, audit the damage. Run this:

In [ ]:
print("=== CUSTOMERS ===")
print(customers.isnull().sum())
print(f"\nTotal rows: {len(customers)}")

print("\n=== ORDERS ===")
print(orders.isnull().sum())
print(f"\nTotal rows: {len(orders)}")

print("\n=== PRODUCTS ===")
print(products.isnull().sum())
print(f"\nTotal rows: {len(products)}")

**Handling nulls — the three strategies**

Not every null is handled the same way. The strategy depends on the business context.

**Strategy 1 — Drop the row**
Use when the record is useless without that value.

A customer with no name and no email cannot be contacted or identified. Drop it.

In [ ]:
print(f"Before Drop:{len(customers)}")

customers_clean = customers.dropna(subset=["name","email"])
print(f"After drop:{len(customers_clean)}")
print(f"Dropped: {len(customers) - len(customers_clean)}")


**Strategy 2 — Fill with a default**

Use when a sensible default exists.

Missing total_spent could mean zero — the customer registered but never bought anything.

In [ ]:
customers_clean["total_spent"] = customers_clean["total_spent"].fillna(0)

print(customers_clean["total_spent"].isnull().sum())

**Strategy 3 — Fill with a calculated value**

Use when a statistical fill makes more sense than a fixed default. Missing product price filled with the category average:

In [ ]:
products["price"] = products.groupby("category")["price"].transform(
    lambda x: x.fillna(x.mean())
)

print(products["price"].isnull().sum())

**Fixing inconsistent strings — the real cleaning work**

This is where your Module 1 knowledge pays off.

In Pandas you apply string operations to entire columns at once — no loop needed.

In [ ]:
# Check what status values actually exist
print("Status values before cleaning:")
print(customers["status"].value_counts())

In [ ]:
# Strip whitespace and lowercase — fixes Active, ACTIVE, etc
customers_clean["status"] = (
    customers_clean["status"]
    .str.strip()
    .str.lower()
)

print("\nStatus values after cleaning:")
print(customers_clean["status"].value_counts())

In [ ]:
# Flag invalid statuses
valid_statuses = {"active", "inactive"}

invalid_mask = ~customers_clean["status"].isin(valid_statuses)
print(f"\nInvalid status rows: {invalid_mask.sum()}")
print(customers_clean[invalid_mask][["customer_id", "name", "status"]])

In [ ]:
# Replace invalid statuses with NaN then drop
customers_clean["status"] = customers_clean["status"].where(
    customers_clean["status"].isin(valid_statuses),
    other=None
)

customers_clean = customers_clean.dropna(subset=["status"])

print(f"\nFinal row count: {len(customers_clean)}")
print(customers_clean["status"].value_counts())

**Cleaning the name column**

In [ ]:
# Strip whitespace from names

customers_clean["name"] = customers_clean["name"].str.strip()

# Check for empty strings after stripping

empty_name = customers_clean[customers_clean["name"] == ""]
print(f"Empty names after strip {len(empty_name)}")

# Replace empty strings with NaN then drop
customers_clean["name"] = customers_clean["name"].replace("",np.nan)
customers_clean = customers_clean.dropna(subset=["name"])

print(f"Final row count: {len(customers_clean)}")


**Handling duplicates**


In [ ]:
# Check for duplicate customer IDs
print(f"Total Rows: {customers_clean['customer_id']}")

print(f"Unique Customer_ids: {customers_clean['customer_id'].nunique}")

In [ ]:
duplicate_mask = customers_clean.duplicated(
    subset=["customer_id"],
    keep=False
)
print(f"Duplicate rows: {duplicate_mask.sum()}")

keep=False — marks ALL duplicate rows as True, not just the second occurrence.

This lets you see all copies before deciding which to keep.
If duplicates exist:

In [ ]:
# Keep the first occurrence, drop the rest

customers_clean = customers_clean.drop_duplicates(
    subset = ["customer_id"],
    keep = "first"
)

print(f"After dedup: {len(customers_clean)}")

**Now you do it. Clean the orders DataFrame:**

1. Normalise status — strip and lowercase
2. Drop rows where status is not in valid order statuses
    valid = {"pending","shipped","delivered","cancelled","returned"}
3. Fill null amounts with the median amount
    (median is better than mean for money — less affected by outliers)
4. Check how many rows remain after cleaning
5. Print the value_counts of status after cleaning

In [ ]:
print("Orders nulls:")
print(orders.isnull().sum())

print("\nOrders status values:")
print(orders["status"].value_counts())

**Normalise status — strip and lowercase**

In [ ]:
# Check nulls
print("Orders nulls:")
print(orders.isnull().sum())

# Check status values

print("\n Order Status Values")
print(orders["status"].value_counts())

In [ ]:
# Normalize status
order['status'] = order['status'].str.strip().str.lower()
# Keep only valid statuses
valid = {"pending","shipped","delivered","cancelled","returned"}

orders = orders[orders['status'].isin(valid)]

#Fill null amounts with median
amount_median = orders['amount'].median()
orders['amount'] = orders['amount'].fillna(amount_median)

# Check remaining rows
print("\nRows after cleaning:", len(orders))

#Status distribution after cleaning
print("\nCleaned status values:")
print(orders["status"].value_counts())

**Actual Solution**

In [ ]:
# Work only on the properly loaded orders DataFrame
# Never reload separately

# Step 1 — normalise status first
orders["status"] = orders["status"].str.strip().str.lower()

# Step 2 — now filter invalid statuses
valid_statuses = {"pending", "shipped", "delivered",
                  "cancelled", "returned"}

orders_clean = orders[orders["status"].isin(valid_statuses)].copy()

# Step 3 — fill null amounts with median
median_amount = orders_clean["amount"].median()
orders_clean["amount"] = orders_clean["amount"].fillna(median_amount)

# Step 4 — check remaining rows
print(f"Original rows : {len(orders)}")
print(f"Cleaned rows  : {len(orders_clean)}")
print(f"Dropped rows  : {len(orders) - len(orders_clean)}")

# Step 5 — status distribution
print("\nStatus distribution after cleaning:")
print(orders_clean["status"].value_counts())

# Step 6 — confirm no nulls remain
print("\nNull counts after cleaning:")
print(orders_clean.isnull().sum())

'''
Notice `.copy()` after the filter. This creates a proper independent copy of the filtered DataFrame. Without it Pandas gives you a `SettingWithCopyWarning` when you try to modify it — because it is not sure if you are modifying the original or the slice. Always use `.copy()` after filtering when you intend to modify the result.

Run this and tell me:
- How many rows were dropped
- What the status distribution looks like

---

## The cleaning sequence to memorise

Every time you clean any DataFrame, always follow this order:
```
1. Audit    — isnull().sum(), value_counts(), dtypes
2. Normalise — strip, lowercase, replace
3. Filter   — remove invalid values
4. Fill     — handle remaining nulls
5. Dedupe   — remove duplicate rows
6. Types    — convert to correct dtypes
7. Verify   — isnull().sum() again, shape, value_counts()'''


**Section 3.6 — Datetime Handling**

Why dates are critical in DE

Every meaningful business question involves time. Revenue this month vs last month. Orders delivered within SLA. Customers inactive for 90 days. Delivery time by warehouse.

None of these are possible without proper datetime handling.
The most common pipeline bug involving dates — they load as strings and nobody notices until an aggregation produces wrong results.

In [ ]:
print(orders_clean["order_date"].dtype)
print(orders_clean["delivery_date"].dtype)
print(customers_clean["join_date"].dtype)

**pd.to_datetime() — converting strings to dates**


In [ ]:
# If delivery_date is still a string — convert it

orders_clean['delivery_date'] = pd.to_datetime(
    orders_clean['delivery_date'],
    errors = "coerce"
)
print(orders_clean["delivery_date"].dtype)
print(orders_clean["delivery_date"].isnull().sum())

**.dt accessor — extracting date parts**

Once a column is datetime you unlock the .dt accessor.

This is how you extract year, month, day and more:

In [ ]:
# Extract date components

# Extract date components
orders_clean["order_year"]    = orders_clean["order_date"].dt.year
orders_clean["order_month"]   = orders_clean["order_date"].dt.month
orders_clean["order_day"]     = orders_clean["order_date"].dt.day
orders_clean["order_weekday"] = orders_clean["order_date"].dt.day_name()
orders_clean["order_quarter"] = orders_clean["order_date"].dt.quarter

print(orders_clean[[
    "order_id", "order_date", "order_year",
    "order_month", "order_weekday", "order_quarter"
]].head(8))

**Time-based filtering**

In [ ]:
# Orders from 2024 only

order_2024 = orders_clean[
    orders_clean['order_date'].dt.year == 2024
    ]

print(f"2024 Orders: {len(order_2024)}")

# Orders from Q1 2024

orders_q1_2024 = orders_clean[
    (orders_clean["order_date"].dt.year == 2024) &
    (orders_clean['order_date'].dt.quarter == 1)

    ]
print(f"Q1 2024 orders: {len(orders_q1_2024)}")

# Orders between two specific dates
start = pd.Timestamp("2024-01-01")
end = pd.Timestamp("2024-06-30")

orders_h1 = orders_clean[
    orders_clean["order_date"].between(start, end)
]
print(f"H1 2024 orders: {len(orders_h1)}")


**timedelta — date arithmetic**

This is where dates become genuinely powerful. Calculating time between two events:

In [ ]:
# Calculate delivery time in days

orders_clean['delivery_days'] = (
    orders_clean['delivery_date'] - orders_clean['order_date']
    ).dt.days

# Only look at delivered orders
delivered = orders_clean[
    orders_clean['status'] == "delivered"
].copy()

print(f"Average Delivery Days: {delivered['delivery_days'].mean(): .1f}")
print(f"Max Delivery Days: {delivered['delivery_days'].max()}")
print(f"Min Delivery Days: {delivered['delivery_days'].min()}")

# Flag orders that took more than 10 days
delivered["late_delivery"] = delivered["delivery_days"] > 10
print(f"\nLate deliveries: {delivered['late_delivery'].sum()}")
print(f"Late percentage: {delivered['late_delivery'].mean()*100:.1f}%")


**resample() — time-based aggregation**

This is one of the most powerful operations in Pandas for DE work. It groups data by time period automatically:

In [ ]:
# Set order_date as index first — required for resample
orders_ts = orders_clean.set_index("order_date")

# Monthly revenue
monthly_revenue = (
    orders_ts['amount']
    .resample("ME")
    .sum()
    .reset_index()
)

monthly_revenue.columns = ["month", "total_revenue"]
print(monthly_revenue)

In [ ]:
# Weekly
#orders_ts["amount"].resample("W").sum()

# Quarterly
orders_ts["amount"].resample("QE").sum()

# Daily
#orders_ts["amount"].resample("D").sum()

Task 1 — Monthly order count and average amount for 2024What we want:

A table showing each month of 2024 with how many orders were placed and what the average order amount was.